In [1]:
import pandas as pd
import warnings

# Suppress future warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Load the dataset from the uploaded CSV file
file_path = 'E:\SignalModel\Out_7.csv'
data = pd.read_csv(file_path)
data['datetime'] =pd.to_datetime(data['datetime'], yearfirst=True)
data.set_index('datetime', inplace=True)
data.sort_index(ascending=True, inplace=True)

In [2]:
# Column names to process for sigai_outputs
sigai_columns = ['SB', 'MB', 'WB', 'WS', 'MS', 'SS']

# Helper function to process a single column for consecutive ones and include the column name
def process_column_with_position(data, column):
    buckets_with_position = []
    start_time = None
    count = 0
    total_sigai = 0
    for i in range(len(data)):
        if data[column][i] == 1:
            if start_time is None:
                start_time = data.index[i]  # Record the start time of the bucket
            count += 1
            total_sigai += data['sigai_output'][i]  # Summing up the sigai_output values for average calculation
        # End of a bucket condition: encounter a 0 or the end of the data
        if data[column][i] == 0 or i == len(data) - 1:
            if count > 0:
                # Calculate average and store the results when a streak ends
                average = total_sigai / count if count > 0 else 0
                buckets_with_position.append({
                    'datetime': start_time,
                    'average': average,
                    'duration': count,  # Duration in hours
                    'position': column,  # Include the column name (position)
                    'trend': None,  # Placeholder for trend
                    'next-position': None  # Placeholder for next-position
                })
                # Reset variables for the next potential streak
                start_time = None
                count = 0
                total_sigai = 0
    return buckets_with_position

# Initialize a DataFrame to store the buckets with position information
clustered_results_with_position = pd.DataFrame(columns=['datetime', 'average', 'duration', 'position', 'trend', 'next-position'])

# Process each column to find consecutive ones and include position
for column in sigai_columns:
    buckets = process_column_with_position(data, column)
    # Concatenate the results for this column with the main DataFrame
    clustered_results_with_position = pd.concat([clustered_results_with_position, pd.DataFrame(buckets)], ignore_index=True)

# Sort the results by datetime
clustered_results_with_position.sort_values('datetime', inplace=True)

# Save the updated DataFrame with the 'position' column to a new CSV file
final_output_file_path = 'E:\SignalModel\Following-Positions\\final_clustered_processed_sigai_buckets.csv'
clustered_results_with_position.to_csv(final_output_file_path, index=False)

# Provide the path to the saved file
final_output_file_path


'E:\\SignalModel\\Following-Positions\\final_clustered_processed_sigai_buckets.csv'

In [3]:
import pandas as pd
import numpy as np

# Function to calculate the slope of the trend line manually
def calculate_slope(x, y):
    if len(x) == 1:  # If there's only one point, the slope is undefined, we can return None or 0
        return 0
    else:
        # Calculate the slope manually: (n*Σ(xy) - Σx*Σy) / (n*Σ(x^2) - (Σx)^2)
        n = len(x)
        sum_x = np.sum(x)
        sum_y = np.sum(y)
        sum_xy = np.sum(x * y)
        sum_x_squared = np.sum(x ** 2)
        numerator = (n * sum_xy) - (sum_x * sum_y)
        denominator = (n * sum_x_squared) - (sum_x ** 2)
        # Avoid division by zero
        if denominator == 0:
            return 0
        slope = numerator / denominator
        return slope

# Load the datasets
original_data = pd.read_csv('E:\SignalModel\Out_7.csv')
bucketed_data = pd.read_csv('E:\SignalModel\Following-Positions\\final_clustered_processed_sigai_buckets.csv')

# Convert 'datetime' to datetime objects
original_data['datetime'] = pd.to_datetime(original_data['datetime'])
bucketed_data['datetime'] = pd.to_datetime(bucketed_data['datetime'])

# Apply the slope calculation to each bucket
for index, row in bucketed_data.iterrows():
    start_time = row['datetime']
    end_time = start_time + pd.Timedelta(hours=row['duration'])
    # Select the data points that fall within the bucket's duration
    mask = (original_data['datetime'] >= start_time) & (original_data['datetime'] < end_time)
    bucket_points = original_data[mask]

    # Convert datetime to the number of hours since the start of the bucket for the x-values
    x_values = (bucket_points['datetime'] - start_time) / pd.Timedelta(hours=1)
    y_values = bucket_points['sigai_output']

    # Calculate the trend for the bucket
    bucketed_data.at[index, 'trend'] = calculate_slope(x_values, y_values)

# Save the updated DataFrame
bucketed_data.to_csv('E:\SignalModel\Following-Positions\\updated_final_clustered_processed_sigai_buckets.csv', index=False)


In [1]:
import pandas as pd

# Load the dataset
data = pd.read_csv('E:\SignalModel\Following-Positions\\updated_final_clustered_processed_sigai_buckets.csv')

# List of unique positions
positions = data['position'].unique()

# Loop through each position and create a separate dataset
for position in positions:
    subset = data[data['position'] == position]
    subset = subset.drop('position', axis=1)
    subset.to_csv(f'E:\SignalModel\Following-Positions\\{position}_subset.csv', index=False)


In [4]:
import pandas as pd
import numpy as np

# Load the dataset from the uploaded CSV file
file_path = 'E:\\SignalModel\\Out_7.csv'
data = pd.read_csv(file_path)
data['datetime'] = pd.to_datetime(data['datetime'], yearfirst=True)
data.set_index('datetime', inplace=True)
data.sort_index(ascending=True, inplace=True)

# Column names to process for sigai_outputs
sigai_columns = ['SB', 'MB', 'WB', 'WS', 'MS', 'SS']

def process_column_with_position(data, column):
    buckets_with_position = []
    start_time = None
    count = 0
    total_sigai = 0
    sigai_values = []  # List to store sigai_output values for variance calculation

    for i in range(len(data)):
        if data[column][i] == 1:
            if start_time is None:
                start_time = data.index[i]  # Record the start time of the bucket
            count += 1
            total_sigai += data['sigai_output'][i]
            sigai_values.append(data['sigai_output'][i])  # Append sigai_output to list

        # End of a bucket condition: encounter a 0 or the end of the data
        if data[column][i] == 0 or i == len(data) - 1:
            if count > 0:
                average = total_sigai / count if count > 0 else 0
                variance = np.var(sigai_values) if count > 1 else 0  # Calculate variance
                buckets_with_position.append({
                    'datetime': start_time,
                    'average': average,
                    'duration': count,
                    'position': column,
                    'trend': None,
                    'next-position': None,
                    'variance': variance  # Add variance to the output
                })
                # Reset variables for the next potential streak
                start_time = None
                count = 0
                total_sigai = 0
                sigai_values = []  # Reset the list for the next bucket

    return buckets_with_position

# Initialize a DataFrame to store the buckets with position information
clustered_results_with_position = pd.DataFrame(columns=['datetime', 'average', 'duration', 'position', 'trend', 'next-position', 'variance'])

# Process each column to find consecutive ones and include position
for column in sigai_columns:
    buckets = process_column_with_position(data, column)
    clustered_results_with_position = pd.concat([clustered_results_with_position, pd.DataFrame(buckets)], ignore_index=True)

# Sort the results by datetime
clustered_results_with_position.sort_values('datetime', inplace=True)

# Save the updated DataFrame with the 'position' column to a new CSV file
final_output_file_path = 'E:\\SignalModel\\Following-Positions\\final_clustered_processed_sigai_buckets.csv'
clustered_results_with_position.to_csv(final_output_file_path, index=False)

# Provide the path to the saved file
final_output_file_path


'E:\\SignalModel\\Following-Positions\\final_clustered_processed_sigai_buckets.csv'